# Práctica 08 — Tests de hipótesis II: comparar varios grupos

**Curso:** Estadística Aplicada con Python · **Clase 08** · **Tiempo estimado:** 60-90 minutos
**Notebook de clase:** `notebooks_clase/clase08_tests_hipotesis_ii.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una celda con `# TODO`: complétala.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los **datos son sintéticos** (*ejemplos ilustrativos*) y las semillas están fijadas, así que los resultados son reproducibles.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

## Ejercicio 1 — El ANOVA a mano

Tres proveedores entregan piezas y medimos el tiempo de montaje en minutos con 4 piezas de cada uno (*ejemplo ilustrativo*). H0: las tres medias son iguales. Calcula:

- `media_global`,
- `ss_entre` (suma de cuadrados entre grupos) y `ss_dentro` (dentro de los grupos),
- `F` = (`ss_entre` / gl entre) / (`ss_dentro` / gl dentro),
- `p_F` con `stats.f.sf`,
- `eta2` = `ss_entre` / (`ss_entre` + `ss_dentro`).

In [ ]:
proveedores = {"A": np.array([6, 8, 7, 7]),
               "B": np.array([9, 11, 10, 10]),
               "C": np.array([12, 10, 11, 11])}
valores = np.concatenate(list(proveedores.values()))

media_global = None   # TODO
ss_entre = None       # TODO
ss_dentro = None      # TODO
F = None              # TODO
p_F = None            # TODO
eta2 = None           # TODO

# Comprobación automática
assert all(v is not None for v in (media_global, ss_entre, ss_dentro, F, p_F, eta2)), "Completa las líneas marcadas con TODO"
assert np.isclose(media_global, valores.mean()), "la media global es la media de los 12 datos"
assert np.isclose(ss_entre, 34.6667, atol=1e-3), "ss_entre = suma de n_g · (media del grupo − media global)²"
assert np.isclose(ss_dentro, 6.0), "ss_dentro = suma de (dato − media de su grupo)²"
assert np.isclose(F, 26.0), "F = (ss_entre / 2) / (ss_dentro / 9)"
assert np.isclose(p_F, stats.f_oneway(*proveedores.values()).pvalue), "p = stats.f.sf(F, 2, 9)"
assert np.isclose(eta2, 34.6667 / 40.6667, atol=1e-3), "η² = ss_entre / (ss_entre + ss_dentro)"
print(f"F = {F:.2f}   p = {p_F:.5f}   η² = {eta2:.3f}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — ANOVA, supuestos y Tukey con `scipy`

Comparas el tiempo de respuesta (ms) de tres servidores con 25 mediciones cada uno (*ejemplo ilustrativo*). Calcula:

- `res_anova`: el resultado de `stats.f_oneway`,
- `p_levene`: el p-value del test de Levene de igualdad de varianzas,
- `res_tk`: el resultado de `stats.tukey_hsd`,
- `pares`: el **conjunto** de pares `(i, j)` con `i < j` cuyo p-value ajustado de Tukey es < 0,05 (por ejemplo `{(0, 1)}`).

In [ ]:
rng_e2 = np.random.default_rng(2026)
srv_a = rng_e2.normal(50, 8, 25)
srv_b = rng_e2.normal(50, 8, 25)
srv_c = rng_e2.normal(58, 8, 25)

res_anova = None   # TODO
p_levene = None    # TODO
res_tk = None      # TODO
pares = None       # TODO: conjunto de tuplas (i, j) con i < j

# Comprobación automática
assert all(v is not None for v in (res_anova, p_levene, res_tk, pares)), "Completa las líneas marcadas con TODO"
assert np.isclose(res_anova.statistic, 5.9371, atol=1e-3) and np.isclose(res_anova.pvalue, 0.0041, atol=1e-4), "usa stats.f_oneway(srv_a, srv_b, srv_c)"
assert np.isclose(p_levene, 0.0292, atol=1e-3), "usa stats.levene(srv_a, srv_b, srv_c).pvalue"
assert isinstance(pares, set), "pares debe ser un conjunto (set)"
assert pares == {(0, 2)}, "recorre los pares i < j y comprueba res_tk.pvalue[i, j] < 0.05"
print(f"F = {res_anova.statistic:.2f}   p = {res_anova.pvalue:.4f}   p de Levene = {p_levene:.4f}   pares distintos: {pares}")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Tests no paramétricos

Comparas tiempos de resolución de incidencias (con cola larga) de dos equipos y de tres equipos (*ejemplo ilustrativo*). Calcula:

- `p_mw`: el p-value de Mann-Whitney (dos colas) entre `equipo_1` y `equipo_2`,
- `prob_sup`: la estimación de P(X > Y) = U / (n₁ · n₂), donde U es el estadístico devuelto para `equipo_1`,
- `p_welch`: el p-value del t de Welch entre los mismos grupos,
- `p_kw`: el p-value de Kruskal-Wallis entre `equipo_1`, `equipo_2` y `equipo_3`.

In [ ]:
rng_e4 = np.random.default_rng(77)
equipo_1 = rng_e4.lognormal(2.0, 0.9, 18)
equipo_2 = rng_e4.lognormal(2.6, 0.9, 22)
equipo_3 = rng_e4.lognormal(2.3, 0.9, 20)

p_mw = None        # TODO
prob_sup = None    # TODO
p_welch = None     # TODO
p_kw = None        # TODO

# Comprobación automática
assert all(v is not None for v in (p_mw, prob_sup, p_welch, p_kw)), "Completa las líneas marcadas con TODO"
assert np.isclose(p_mw, 0.1313, atol=1e-3), "usa stats.mannwhitneyu(equipo_1, equipo_2, alternative='two-sided')"
assert np.isclose(prob_sup, 142 / (18 * 22)), "prob_sup = U / (18 · 22), con U = statistic del resultado de Mann-Whitney"
assert np.isclose(p_welch, 0.5719, atol=1e-3), "usa stats.ttest_ind(equipo_1, equipo_2, equal_var=False)"
assert np.isclose(p_kw, 0.2694, atol=1e-3), "usa stats.kruskal(equipo_1, equipo_2, equipo_3)"
print(f"Mann-Whitney p = {p_mw:.4f}   P(X > Y) ≈ {prob_sup:.3f}   Welch p = {p_welch:.4f}   Kruskal-Wallis p = {p_kw:.4f}")
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — ¿ANOVA o Kruskal-Wallis? Datos de cola larga

Los tiempos de resolución de tickets (minutos) de tres turnos, con 15 tickets cada uno, tienen cola larga (*ejemplo ilustrativo*). Calcula:

- `p_anova` y `p_kw`: los p-values de `stats.f_oneway` y de `stats.kruskal` sobre `turno_1`, `turno_2` y `turno_3`;
- dentro del bucle de simulación (1 000 estudios con las mismas medias de la función `generar`), suma 1 a `rech_anova` cuando el ANOVA rechaza (p < 0,05) y 1 a `rech_kw` cuando Kruskal-Wallis rechaza;
- `mejor`: `"anova"` o `"kruskal"`, el test con mayor potencia estimada en esta situación.

In [ ]:
def generar(r):
    return [r.lognormal(m, 1.0, 15) for m in (0, 0.5, 1.0)]

turno_1, turno_2, turno_3 = generar(np.random.default_rng(29))
p_anova = None    # TODO
p_kw = None       # TODO

REPS = 1000
rng_sim = np.random.default_rng(99)
rech_anova = 0
rech_kw = 0
for _ in range(REPS):
    gs = generar(rng_sim)
    # TODO: actualiza rech_anova y rech_kw
pot_anova = rech_anova / REPS
pot_kw = rech_kw / REPS
mejor = None      # TODO: "anova" o "kruskal"

# Comprobación automática
assert p_anova is not None and p_kw is not None and mejor is not None and rech_anova + rech_kw > 0, "Completa las líneas marcadas con TODO"
assert np.isclose(p_anova, 0.0594, atol=1e-3), "usa stats.f_oneway(turno_1, turno_2, turno_3).pvalue"
assert np.isclose(p_kw, 0.0028, atol=1e-3), "usa stats.kruskal(turno_1, turno_2, turno_3).pvalue"
assert abs(pot_anova - 0.454) < 0.035 and abs(pot_kw - 0.657) < 0.035, "en cada estudio: rech_anova += stats.f_oneway(*gs).pvalue < 0.05 (y lo mismo con kruskal)"
assert mejor == "kruskal", "con colas largas Kruskal-Wallis tiene más potencia que el ANOVA"
print(f"p ANOVA = {p_anova:.4f}   p Kruskal-Wallis = {p_kw:.4f}   potencia: ANOVA {pot_anova:.3f}, Kruskal-Wallis {pot_kw:.3f}")
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Leer un resultado: F, p, η² y post-hoc

**Parte A.** Un informe dice, sin más datos: «ANOVA de un factor, F(3, 56) = 4,20». Calcula:

- `p_inf`: el p-value correspondiente (`stats.f.sf`),
- `eta2_inf`: η² a partir de F y los grados de libertad, η² = F · gl_entre / (F · gl_entre + gl_dentro),
- `tamano`: `"pequeño"` (η² < 0,06), `"mediano"` (0,06 ≤ η² < 0,14) o `"grande"` (η² ≥ 0,14).

**Parte B.** Cuatro turnos con 15 tickets cada uno (*ejemplo ilustrativo*). Calcula:

- `res_tk`: el resultado de `stats.tukey_hsd` sobre los cuatro turnos,
- `pares_ic`: el **conjunto** de pares `(i, j)` con `i < j` cuyo intervalo de confianza al 95 % de Tukey (`res_tk.confidence_interval()`, atributos `low` y `high`) **no** contiene el 0,
- `interpretacion`: la única redacción correcta, `"a"`, `"b"` o `"c"`:
  - `"a"`: los cuatro turnos difieren entre sí, porque el ANOVA es significativo.
  - `"b"`: no todas las medias son iguales; según Tukey el turno 4 es más lento que los turnos 1 y 2, y en el resto de pares no hay diferencias distinguibles del ruido.
  - `"c"`: el p-value del ANOVA es la probabilidad de que las cuatro medias sean iguales.

In [ ]:
F_inf, gl_entre, gl_dentro = 4.20, 3, 56
p_inf = None       # TODO
eta2_inf = None    # TODO
tamano = None      # TODO: "pequeño", "mediano" o "grande"

rng_e5 = np.random.default_rng(14)
turnos5 = [rng_e5.normal(m, 8, 15) for m in (30, 31, 33, 39)]
res_tk = None            # TODO
pares_ic = None          # TODO: conjunto de tuplas (i, j) con i < j
interpretacion = None    # TODO: "a", "b" o "c"

# Comprobación automática
assert all(v is not None for v in (p_inf, eta2_inf, tamano, res_tk, pares_ic, interpretacion)), "Completa las líneas marcadas con TODO"
assert np.isclose(p_inf, 0.0095, atol=1e-4), "p = stats.f.sf(F_inf, gl_entre, gl_dentro)"
assert np.isclose(eta2_inf, 0.1837, atol=1e-3), "η² = F · gl_entre / (F · gl_entre + gl_dentro)"
assert tamano == "grande", "compara η² con 0,06 y 0,14"
assert isinstance(pares_ic, set) and pares_ic == {(0, 3), (1, 3)}, "recorre los pares i < j y comprueba si ic.low[i, j] > 0 o ic.high[i, j] < 0"
assert interpretacion == "b", "un ANOVA significativo no dice qué grupos difieren, y el p-value no es la probabilidad de H0"
print(f"p = {p_inf:.4f}   η² = {eta2_inf:.3f} ({tamano})   pares con IC sin el 0: {sorted(pares_ic)}")
print("Ejercicio 5 correcto ✔")

---
## Reto final — Un informe de comparación de grupos

Un equipo compara el tiempo de gestión de tickets (minutos) de 4 turnos con 12 tickets cada uno (*ejemplo ilustrativo*). Completa la función `informe_grupos(grupos)`, que recibe una lista de arrays y devuelve un diccionario con las claves `"F"`, `"p"` (del ANOVA de un factor), `"eta2"`, `"p_levene"`, `"n_pares"` (número de pares con p ajustado de Tukey < 0,05) y `"p_kw"` (p-value de Kruskal-Wallis).

Después elige en `respuesta` la única interpretación correcta:

- `"a"`: un ANOVA significativo significa que todos los grupos difieren entre sí.
- `"b"`: un ANOVA significativo solo indica que al menos un grupo difiere; para saber cuáles hace falta una comparación posterior controlada (Tukey) y no varios t-tests sin corregir. Y si p no es significativo, con solo 12 datos por grupo no se puede descartar un efecto.
- `"c"`: si el ANOVA da p = 0,04, la probabilidad de que H0 sea cierta es del 4 %.

In [ ]:
def informe_grupos(grupos):
    """Devuelve F, p, eta2, p_levene, n_pares (Tukey) y p_kw."""
    # TODO: ANOVA, η², Levene, Tukey y Kruskal-Wallis
    return None

rng_r = np.random.default_rng(910)
turnos = [rng_r.normal(mu, 7, 12) for mu in (40, 41, 42, 50)]
inf = informe_grupos(turnos)
respuesta = None    # TODO: "a", "b" o "c"

# Comprobación automática
assert inf is not None and respuesta is not None, "Completa la función y la respuesta (TODO)"
assert set(inf) == {"F", "p", "eta2", "p_levene", "n_pares", "p_kw"}, "el diccionario debe tener exactamente las claves F, p, eta2, p_levene, n_pares y p_kw"
assert np.isclose(inf["F"], 2.5523, atol=1e-3) and np.isclose(inf["p"], 0.0676, atol=1e-3), "usa stats.f_oneway(*grupos)"
assert np.isclose(inf["p_levene"], 0.7112, atol=1e-3), "usa stats.levene(*grupos)"
assert np.isclose(inf["p_kw"], 0.0736, atol=1e-3), "usa stats.kruskal(*grupos).pvalue"
todo = np.concatenate(turnos)
assert np.isclose(inf["eta2"], sum(len(g) * (g.mean() - todo.mean()) ** 2 for g in turnos) / ((todo - todo.mean()) ** 2).sum()), "η² = SS entre / SS total"
assert inf["n_pares"] == 0, "cuenta los pares i < j con res_tk.pvalue[i, j] < 0.05"
assert respuesta == "b", "un ANOVA no dice qué grupos difieren, y un p alto no demuestra ausencia de efecto"
print(f"F = {inf['F']:.2f}   p = {inf['p']:.4f}   η² = {inf['eta2']:.3f}   p de Levene = {inf['p_levene']:.3f}   pares distintos = {inf['n_pares']}   p Kruskal-Wallis = {inf['p_kw']:.4f}")
print("Reto final correcto ✔")

---
## Conclusión de la práctica

Has calculado un ANOVA a mano, lo has completado con supuestos y comparaciones de Tukey, has usado tests no paramétricos, has comparado ANOVA y Kruskal-Wallis con datos de cola larga y has leído un resultado completo (F, p, η², post-hoc). Siguiente paso: la clase 9 (categorías y comparaciones múltiples).

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
proveedores = {"A": np.array([6, 8, 7, 7]),
               "B": np.array([9, 11, 10, 10]),
               "C": np.array([12, 10, 11, 11])}
valores = np.concatenate(list(proveedores.values()))

media_global = valores.mean()
ss_entre = sum(len(v) * (v.mean() - media_global) ** 2 for v in proveedores.values())
ss_dentro = sum(((v - v.mean()) ** 2).sum() for v in proveedores.values())
F = (ss_entre / 2) / (ss_dentro / 9)
p_F = stats.f.sf(F, 2, 9)
eta2 = ss_entre / (ss_entre + ss_dentro)

assert all(v is not None for v in (media_global, ss_entre, ss_dentro, F, p_F, eta2)), "Completa las líneas marcadas con TODO"
assert np.isclose(media_global, valores.mean()), "la media global es la media de los 12 datos"
assert np.isclose(ss_entre, 34.6667, atol=1e-3), "ss_entre = suma de n_g · (media del grupo − media global)²"
assert np.isclose(ss_dentro, 6.0), "ss_dentro = suma de (dato − media de su grupo)²"
assert np.isclose(F, 26.0), "F = (ss_entre / 2) / (ss_dentro / 9)"
assert np.isclose(p_F, stats.f_oneway(*proveedores.values()).pvalue), "p = stats.f.sf(F, 2, 9)"
assert np.isclose(eta2, 34.6667 / 40.6667, atol=1e-3), "η² = ss_entre / (ss_entre + ss_dentro)"
print(f"F = {F:.2f}   p = {p_F:.5f}   η² = {eta2:.3f}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
rng_e2 = np.random.default_rng(2026)
srv_a = rng_e2.normal(50, 8, 25)
srv_b = rng_e2.normal(50, 8, 25)
srv_c = rng_e2.normal(58, 8, 25)

res_anova = stats.f_oneway(srv_a, srv_b, srv_c)
p_levene = stats.levene(srv_a, srv_b, srv_c).pvalue
res_tk = stats.tukey_hsd(srv_a, srv_b, srv_c)
pares = {(i, j) for i in range(3) for j in range(i + 1, 3) if res_tk.pvalue[i, j] < 0.05}

assert all(v is not None for v in (res_anova, p_levene, res_tk, pares)), "Completa las líneas marcadas con TODO"
assert np.isclose(res_anova.statistic, 5.9371, atol=1e-3) and np.isclose(res_anova.pvalue, 0.0041, atol=1e-4), "usa stats.f_oneway(srv_a, srv_b, srv_c)"
assert np.isclose(p_levene, 0.0292, atol=1e-3), "usa stats.levene(srv_a, srv_b, srv_c).pvalue"
assert isinstance(pares, set), "pares debe ser un conjunto (set)"
assert pares == {(0, 2)}, "recorre los pares i < j y comprueba res_tk.pvalue[i, j] < 0.05"
print(f"F = {res_anova.statistic:.2f}   p = {res_anova.pvalue:.4f}   p de Levene = {p_levene:.4f}   pares distintos: {pares}")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
rng_e4 = np.random.default_rng(77)
equipo_1 = rng_e4.lognormal(2.0, 0.9, 18)
equipo_2 = rng_e4.lognormal(2.6, 0.9, 22)
equipo_3 = rng_e4.lognormal(2.3, 0.9, 20)

res_mw = stats.mannwhitneyu(equipo_1, equipo_2, alternative="two-sided")
p_mw = res_mw.pvalue
prob_sup = res_mw.statistic / (len(equipo_1) * len(equipo_2))
p_welch = stats.ttest_ind(equipo_1, equipo_2, equal_var=False).pvalue
p_kw = stats.kruskal(equipo_1, equipo_2, equipo_3).pvalue

assert all(v is not None for v in (p_mw, prob_sup, p_welch, p_kw)), "Completa las líneas marcadas con TODO"
assert np.isclose(p_mw, 0.1313, atol=1e-3), "usa stats.mannwhitneyu(equipo_1, equipo_2, alternative='two-sided')"
assert np.isclose(prob_sup, 142 / (18 * 22)), "prob_sup = U / (18 · 22), con U = statistic del resultado de Mann-Whitney"
assert np.isclose(p_welch, 0.5719, atol=1e-3), "usa stats.ttest_ind(equipo_1, equipo_2, equal_var=False)"
assert np.isclose(p_kw, 0.2694, atol=1e-3), "usa stats.kruskal(equipo_1, equipo_2, equipo_3)"
print(f"Mann-Whitney p = {p_mw:.4f}   P(X > Y) ≈ {prob_sup:.3f}   Welch p = {p_welch:.4f}   Kruskal-Wallis p = {p_kw:.4f}")
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def generar(r):
    return [r.lognormal(m, 1.0, 15) for m in (0, 0.5, 1.0)]

turno_1, turno_2, turno_3 = generar(np.random.default_rng(29))
p_anova = stats.f_oneway(turno_1, turno_2, turno_3).pvalue
p_kw = stats.kruskal(turno_1, turno_2, turno_3).pvalue

REPS = 1000
rng_sim = np.random.default_rng(99)
rech_anova = 0
rech_kw = 0
for _ in range(REPS):
    gs = generar(rng_sim)
    rech_anova += stats.f_oneway(*gs).pvalue < 0.05
    rech_kw += stats.kruskal(*gs).pvalue < 0.05
pot_anova = rech_anova / REPS
pot_kw = rech_kw / REPS
mejor = "kruskal" if pot_kw > pot_anova else "anova"

assert p_anova is not None and p_kw is not None and mejor is not None and rech_anova + rech_kw > 0, "Completa las líneas marcadas con TODO"
assert np.isclose(p_anova, 0.0594, atol=1e-3), "usa stats.f_oneway(turno_1, turno_2, turno_3).pvalue"
assert np.isclose(p_kw, 0.0028, atol=1e-3), "usa stats.kruskal(turno_1, turno_2, turno_3).pvalue"
assert abs(pot_anova - 0.454) < 0.035 and abs(pot_kw - 0.657) < 0.035, "en cada estudio: rech_anova += stats.f_oneway(*gs).pvalue < 0.05 (y lo mismo con kruskal)"
assert mejor == "kruskal", "con colas largas Kruskal-Wallis tiene más potencia que el ANOVA"
print(f"p ANOVA = {p_anova:.4f}   p Kruskal-Wallis = {p_kw:.4f}   potencia: ANOVA {pot_anova:.3f}, Kruskal-Wallis {pot_kw:.3f}")
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
F_inf, gl_entre, gl_dentro = 4.20, 3, 56
p_inf = stats.f.sf(F_inf, gl_entre, gl_dentro)
eta2_inf = F_inf * gl_entre / (F_inf * gl_entre + gl_dentro)
tamano = "pequeño" if eta2_inf < 0.06 else ("mediano" if eta2_inf < 0.14 else "grande")

rng_e5 = np.random.default_rng(14)
turnos5 = [rng_e5.normal(m, 8, 15) for m in (30, 31, 33, 39)]
res_tk = stats.tukey_hsd(*turnos5)
ic = res_tk.confidence_interval()
pares_ic = {(i, j) for i in range(4) for j in range(i + 1, 4) if ic.low[i, j] > 0 or ic.high[i, j] < 0}
interpretacion = "b"

assert all(v is not None for v in (p_inf, eta2_inf, tamano, res_tk, pares_ic, interpretacion)), "Completa las líneas marcadas con TODO"
assert np.isclose(p_inf, 0.0095, atol=1e-4), "p = stats.f.sf(F_inf, gl_entre, gl_dentro)"
assert np.isclose(eta2_inf, 0.1837, atol=1e-3), "η² = F · gl_entre / (F · gl_entre + gl_dentro)"
assert tamano == "grande", "compara η² con 0,06 y 0,14"
assert isinstance(pares_ic, set) and pares_ic == {(0, 3), (1, 3)}, "recorre los pares i < j y comprueba si ic.low[i, j] > 0 o ic.high[i, j] < 0"
assert interpretacion == "b", "un ANOVA significativo no dice qué grupos difieren, y el p-value no es la probabilidad de H0"
print(f"p = {p_inf:.4f}   η² = {eta2_inf:.3f} ({tamano})   pares con IC sin el 0: {sorted(pares_ic)}")
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def informe_grupos(grupos):
    """Devuelve F, p, eta2, p_levene, n_pares (Tukey) y p_kw."""
    res = stats.f_oneway(*grupos)
    todo = np.concatenate(grupos)
    ss_total = ((todo - todo.mean()) ** 2).sum()
    ss_e = sum(len(g) * (g.mean() - todo.mean()) ** 2 for g in grupos)
    tk = stats.tukey_hsd(*grupos)
    k = len(grupos)
    n_pares = sum(1 for i in range(k) for j in range(i + 1, k) if tk.pvalue[i, j] < 0.05)
    return {"F": res.statistic, "p": res.pvalue, "eta2": ss_e / ss_total,
            "p_levene": stats.levene(*grupos).pvalue, "n_pares": n_pares,
            "p_kw": stats.kruskal(*grupos).pvalue}

rng_r = np.random.default_rng(910)
turnos = [rng_r.normal(mu, 7, 12) for mu in (40, 41, 42, 50)]
inf = informe_grupos(turnos)
respuesta = "b"

assert inf is not None and respuesta is not None, "Completa la función y la respuesta (TODO)"
assert set(inf) == {"F", "p", "eta2", "p_levene", "n_pares", "p_kw"}, "el diccionario debe tener exactamente las claves F, p, eta2, p_levene, n_pares y p_kw"
assert np.isclose(inf["F"], 2.5523, atol=1e-3) and np.isclose(inf["p"], 0.0676, atol=1e-3), "usa stats.f_oneway(*grupos)"
assert np.isclose(inf["p_levene"], 0.7112, atol=1e-3), "usa stats.levene(*grupos)"
assert np.isclose(inf["p_kw"], 0.0736, atol=1e-3), "usa stats.kruskal(*grupos).pvalue"
todo = np.concatenate(turnos)
assert np.isclose(inf["eta2"], sum(len(g) * (g.mean() - todo.mean()) ** 2 for g in turnos) / ((todo - todo.mean()) ** 2).sum()), "η² = SS entre / SS total"
assert inf["n_pares"] == 0, "cuenta los pares i < j con res_tk.pvalue[i, j] < 0.05"
assert respuesta == "b", "un ANOVA no dice qué grupos difieren, y un p alto no demuestra ausencia de efecto"
print(f"F = {inf['F']:.2f}   p = {inf['p']:.4f}   η² = {inf['eta2']:.3f}   p de Levene = {inf['p_levene']:.3f}   pares distintos = {inf['n_pares']}   p Kruskal-Wallis = {inf['p_kw']:.4f}")
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (F entre/dentro, post-hoc, qué contrasta Mann-Whitney, ANOVA frente a Kruskal-Wallis, cómo se lee un informe)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*